In [33]:
# ─────────────────────────────────────────────────────────────
# CELL 1: SETUP & LIBRARIES
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

print("✅ Cell 1 Complete: All libraries loaded.")

✅ Cell 1 Complete: All libraries loaded.


In [34]:
# ─────────────────────────────────────────────────────────────
# CELL 2: UNIVERSAL KINEMATIC FEATURE ENGINEERING (N-DOF)
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ================== CONFIGURATION ==================
# Change this single number to process your different datasets!
DOF = 7
FILE_NAME = f"{DOF}dof_500.csv"

df = pd.read_csv(FILE_NAME)
print(f"Loaded {FILE_NAME}. Original rows: {len(df)}")

# ================== 1. TRAJECTORY FILTERING ==================
successful_episodes = df[df["is_success"] == 1]["episode"].unique()
df = df[df["episode"].isin(successful_episodes)].copy()
print(f"Filtered for successful episodes. Remaining rows: {len(df)}")

df = df.sort_values(["episode", "step_count"])

# ================== 2. SPATIAL FEATURES ==================
df["ee_error_x"] = df["ent_0_x"] - 0.5
df["ee_error_y"] = df["ent_0_y"] - 0.0
df["ee_error_z"] = df["ent_0_z"] - 0.5
df["ee_dist"] = np.sqrt(df["ee_error_x"]**2 + df["ee_error_y"]**2 + df["ee_error_z"]**2)

# ================== 3. DYNAMIC KINEMATIC FEATURES ==================
episode_len = df.groupby("episode").size()
df["episode_len"] = df["episode"].map(episode_len)
df["progress"] = df["step_count"] / df["episode_len"]

for i in range(DOF):
    # Trigonometric Encodings
    df[f"sin_joint_{i}"] = np.sin(df[f"joint_{i}"])
    df[f"cos_joint_{i}"] = np.cos(df[f"joint_{i}"])

    # Velocity
    df[f"vel_{i}"] = df[f"joint_{i}"] - df.groupby("episode")[f"joint_{i}"].shift(1)
    df[f"vel_{i}"] = df[f"vel_{i}"].fillna(0)

    # Acceleration
    df[f"accel_{i}"] = df[f"vel_{i}"] - df.groupby("episode")[f"vel_{i}"].shift(1)
    df[f"accel_{i}"] = df[f"accel_{i}"].fillna(0)

    # Memory (Previous Actions)
    df[f"prev_action_{i}"] = df.groupby("episode")[f"action_{i}"].shift(1).fillna(0)

    # Targets (Deltas)
    df[f"delta_action_{i}"] = df[f"action_{i}"] - df[f"prev_action_{i}"]

# ================== 4. DYNAMIC INPUTS & OUTPUTS ==================
# Automatically generate the list of outputs based on DOF
OUTPUT_COLS = [f"delta_action_{i}" for i in range(DOF)]

# Automatically exclude all raw actions and raw joints based on DOF
raw_actions = [f"action_{i}" for i in range(DOF)]
raw_joints = [f"joint_{i}" for i in range(DOF)]

exclude_cols = OUTPUT_COLS + raw_actions + raw_joints + [
    'episode', 'step_count', 'reward', 'done', 'is_success',
    'episode_len', 'ent_0_x', 'ent_0_y', 'ent_0_z'
]

INPUT_ALL = [col for col in df.columns if col not in exclude_cols]

X = df[INPUT_ALL].copy()
y = df[OUTPUT_COLS].copy()

print(f"✅ Cell 2 Complete. Pipeline configured for {DOF}-DOF robot.")
print(f"Total input features: {len(INPUT_ALL)}")
print(f"Total outputs being predicted: {len(OUTPUT_COLS)}")

Loaded 7dof_500.csv. Original rows: 49120
Filtered for successful episodes. Remaining rows: 1620
✅ Cell 2 Complete. Pipeline configured for 7-DOF robot.
Total input features: 40
Total outputs being predicted: 7


In [35]:
# ─────────────────────────────────────────────────────────────
# CELL 4: DATA SPLITTING & SCALING
# ─────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Cell 4 Complete: Data is split and standardized.")

✅ Cell 4 Complete: Data is split and standardized.


In [36]:
# ─────────────────────────────────────────────────────────────
# CELL 5: TRAIN 6 MODELS (LR, SVM, RF, XGB, MLP, KNN)
# ─────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

print("Training all models side-by-side...\n")

# 1. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
print("- Linear Regression trained.")

# 2. Support Vector Machine
svm_base = SVR(kernel='rbf', C=1.0, gamma='scale')
svm_model = MultiOutputRegressor(svm_base)
svm_model.fit(X_train_scaled, y_train)
print("- SVM trained.")

# # 3. Random Forest
# rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
# rf_model = MultiOutputRegressor(rf_base)
# rf_model.fit(X_train_scaled, y_train)
# print("- Random Forest trained.")

# 4. XGBoost
xgb_base = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
xgb_model = MultiOutputRegressor(xgb_base)
xgb_model.fit(X_train_scaled, y_train)
print("- XGBoost trained.")

print("\n✅ Cell 5 Complete.")

Training all models side-by-side...

- Linear Regression trained.
- SVM trained.
- XGBoost trained.

✅ Cell 5 Complete.


In [37]:
# ─────────────────────────────────────────────────────────────
# CELL 6: THE FINAL REPORT CARD (ALL 4 MODELS)
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import r2_score

def evaluate_model(model_name, model):
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)

    print(f"--- {model_name} ---")
    print(f"  R-Squared Score : {r2:.4f}\n")

print(f"=== {DOF}DOF ===\n")
evaluate_model("Linear Regression", lr_model)
evaluate_model("Support Vector Machine", svm_model)
# evaluate_model("Random Forest", rf_model)
evaluate_model("XGBoost", xgb_model)

=== 7DOF ===

--- Linear Regression ---
  R-Squared Score : 0.4803

--- Support Vector Machine ---
  R-Squared Score : 0.3960

--- XGBoost ---
  R-Squared Score : 0.4018

